<h1> Data Analysis </h1>

<h2> Load Dataset </h2>

In [1]:
import sys
from pathlib import Path

# Project root: works if kernel cwd is repo root or notebooks/
_root = Path.cwd()
if not (_root / "data" / "load_dataset.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from data.load_dataset import DatasetLoader

loader = DatasetLoader()

<h2> Check Sarcastic ratio </h2>

<p>Label balance (sarcastic vs not), share of rows by publisher (from <code>article_link</code>), and headline duplication.</p>

In [3]:
import pandas as pd
from urllib.parse import urlparse

# Use path from loader (stable across versions). Avoids stale-kernel errors if
# `get_dataframe` was added later — restart kernel after pulling code if you prefer loader.get_dataframe().
df = pd.read_json(loader.dataset_path, lines=True, encoding="utf-8")
n = len(df)


def publisher_from_url(url: str) -> str:
    if not isinstance(url, str) or not url.strip():
        return "unknown"
    host = (urlparse(url).hostname or "").lower()
    if "onion" in host:
        return "theonion"
    if "huffingtonpost" in host or "huffpost" in host:
        return "huffpost"
    return "other"


df = df.copy()
df["publisher"] = df["article_link"].map(publisher_from_url)

# --- Sarcastic vs non-sarcastic (is_sarcastic: 1 = sarcastic, 0 = not) ---
label_counts = df["is_sarcastic"].value_counts().sort_index()
label_pct = df["is_sarcastic"].value_counts(normalize=True).sort_index() * 100

print("=== Labels (is_sarcastic) ===")
print(f"Total rows: {n}")
for lab, c in label_counts.items():
    name = "sarcastic" if lab == 1 else "non-sarcastic"
    print(f"  {name} ({lab}): {c} ({label_pct[lab]:.2f}%)")

# --- By website / publisher ---
print("\n=== Rows by publisher (from article_link host) ===")
pub_counts = df["publisher"].value_counts()
pub_pct = df["publisher"].value_counts(normalize=True) * 100
for pub, c in pub_counts.items():
    print(f"  {pub}: {c} ({pub_pct[pub]:.2f}%)")

# --- Repeated headlines ---
n_unique_headlines = df["headline"].nunique()
dup_row_mask = df.duplicated(subset=["headline"], keep="first")
n_rows_duplicate_headline = int(dup_row_mask.sum())
headlines_with_repeats = (df.groupby("headline").size() > 1).sum()

print("\n=== Headline duplication ===")
print(f"  Unique headlines: {n_unique_headlines}")
print(f"  Headline strings that appear more than once: {int(headlines_with_repeats)}")
print(f"  Rows that repeat an earlier headline (extra copies): {n_rows_duplicate_headline}")

# Publisher vs label (v2: HuffPost ↔ non-sarcastic, Onion ↔ sarcastic — spurious cue risk)
print("\n=== Publisher × is_sarcastic (crosstab) ===")
print(pd.crosstab(df["publisher"], df["is_sarcastic"], margins=True))

=== Labels (is_sarcastic) ===
Total rows: 28619
  non-sarcastic (0): 14985 (52.36%)
  sarcastic (1): 13634 (47.64%)

=== Rows by publisher (from article_link host) ===
  huffpost: 14985 (52.36%)
  theonion: 13634 (47.64%)

=== Headline duplication ===
  Unique headlines: 28503
  Headline strings that appear more than once: 80
  Rows that repeat an earlier headline (extra copies): 116

=== Publisher × is_sarcastic (crosstab) ===
is_sarcastic      0      1    All
publisher                        
huffpost      14985      0  14985
theonion          0  13634  13634
All           14985  13634  28619


<h3> sarcastic label ratio </h3>
Total rows: 28619 <br>
non-sarcastic (0): 14985 (52.36%) <br>
sarcastic (1): 13634 (47.64%) <br>
the ratio of non-sarcastic headlines outweights the saracastic version by a tiny bit. And the ratio highly coincident with the websites. <br>
Therefore all saracastics ones are coming from the the onion and non-sarcastic ones coming from the huffpost <br>

There are multiple repeated headlines so we need to remove them and perform the ratio check again.

In [4]:
# --- Deeper look: duplicate headline patterns ---
counts_per_headline = df.groupby("headline").size()
repeated = counts_per_headline[counts_per_headline > 1]

print("=== How many times do repeated headlines appear? ===")
print(repeated.value_counts().sort_index().rename("num_headlines").to_string())
print(
    f"\n({len(repeated)} distinct headline strings account for "
    f"{repeated.sum()} rows → {int(repeated.sum() - len(repeated))} extra rows)"
)

# Do duplicate rows ever disagree on label? (quality check)
n_mixed = (
    df.groupby("headline", sort=False)["is_sarcastic"].nunique().gt(1).sum()
)
print(f"\nHeadlines with conflicting is_sarcastic across rows: {int(n_mixed)}")

# --- Ratio after removing duplicate headline rows (keep first occurrence) ---
df_dedup = df.drop_duplicates(subset=["headline"], keep="first")
n_dedup = len(df_dedup)

print("\n=== Ratios after drop_duplicates(headline), keep='first' ===")
print(f"Rows: {n_dedup} (removed {n - n_dedup} duplicate rows)")

ld = df_dedup["is_sarcastic"].value_counts().sort_index()
pd_pct = df_dedup["is_sarcastic"].value_counts(normalize=True).sort_index() * 100
print("\nLabels (is_sarcastic):")
for lab, c in ld.items():
    name = "sarcastic" if lab == 1 else "non-sarcastic"
    print(f"  {name} ({lab}): {c} ({pd_pct[lab]:.2f}%)")

print("\nPublisher:")
for pub, c in df_dedup["publisher"].value_counts().items():
    p = 100 * c / n_dedup
    print(f"  {pub}: {c} ({p:.2f}%)")

print("\nPublisher × is_sarcastic:")
print(pd.crosstab(df_dedup["publisher"], df_dedup["is_sarcastic"], margins=True))

# Sample of most-repeated headlines (for manual inspection)
print("\n=== Top repeated headlines (by row count) ===")
top_rep = repeated.sort_values(ascending=False).head(12)
for headline, k in top_rep.items():
    short = (headline[:77] + "…") if len(headline) > 80 else headline
    print(f"  ×{k}  {short}")

=== How many times do repeated headlines appear? ===
2     72
3      2
4      2
6      1
10     2
12     1

(80 distinct headline strings account for 196 rows → 116 extra rows)

Headlines with conflicting is_sarcastic across rows: 0

=== Ratios after drop_duplicates(headline), keep='first' ===
Rows: 28503 (removed 116 duplicate rows)

Labels (is_sarcastic):
  non-sarcastic (0): 14951 (52.45%)
  sarcastic (1): 13552 (47.55%)

Publisher:
  huffpost: 14951 (52.45%)
  theonion: 13552 (47.55%)

Publisher × is_sarcastic:
is_sarcastic      0      1    All
publisher                        
huffpost      14951      0  14951
theonion          0  13552  13552
All           14951  13552  28503

=== Top repeated headlines (by row count) ===
  ×12  'no way to prevent this,' says only nation where this regularly happens
  ×10  sunday roundup
  ×10  the 20 funniest tweets from women this week
  ×6  the funniest tweets from parents this week
  ×4  report: make it stop
  ×4  the funniest tweets from wom

After removing exact duplicate headlines, the dataset remains reasonably balanced (52.45% non-sarcastic, 47.55% sarcastic), so standard stratified splitting is sufficient without resampling. However, publisher is perfectly aligned with label in this corpus, making source bias a much more serious concern than class imbalance.